# 第 6 章 · 前向链与后向链

[章节网页](../ch6.html)

## 运行内容

- 加载规则与事实
- 运行前向链
- 运行后向链

## 0. 环境与数据

In [1]:
# 准备运行时：本 notebook 内嵌所需源码和数据，不依赖在线封装文件。
import importlib.util
import subprocess
import sys
from pathlib import Path

INLINE_RUNTIME_FILES = {
  "labs/ch06/reasoning.py": "\"\"\"Chapter 6 reasoning demos — aligned with ch6.js.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\n\nCOMMON = Path(__file__).resolve().parent.parent / \"common\"\n\n\ndef load_rules():\n    with (COMMON / \"ch6_rules.json\").open(encoding=\"utf-8\") as f:\n        return json.load(f)\n\n\ndef load_kg():\n    with (COMMON / \"ch6_kg.json\").open(encoding=\"utf-8\") as f:\n        return json.load(f)\n\n\ndef facts_goal_table() -> pd.DataFrame:\n    data = load_rules()\n    return pd.DataFrame(\n        [\n            {\"项目\": \"初始事实\", \"内容\": \"；\".join(data[\"facts\"])},\n            {\"项目\": \"证明目标\", \"内容\": data[\"goal\"]},\n        ]\n    )\n\n\ndef rules_table() -> pd.DataFrame:\n    data = load_rules()\n    return pd.DataFrame(\n        [\n            {\"规则\": rule[\"id\"], \"IF\": \" AND \".join(rule[\"if\"]), \"THEN\": rule[\"then\"]}\n            for rule in data[\"rules\"]\n        ]\n    )\n\n\ndef forward_chain(max_steps: int = 10) -> list[str]:\n    data = load_rules()\n    known = set(data[\"facts\"])\n    log = [f\"初始事实: {', '.join(sorted(known))}\"]\n    for step in range(1, max_steps + 1):\n        added = []\n        for rule in data[\"rules\"]:\n            cond = rule[\"if\"][0]\n            pred = cond.split(\"(\")[0]\n            var = cond[cond.index(\"(\") + 1 : cond.index(\")\")]\n            inst = cond.replace(f\"({var})\", \"(苏格拉底)\")\n            if inst in known:\n                concl = rule[\"then\"].replace(f\"({var})\", \"(苏格拉底)\")\n                if concl not in known:\n                    known.add(concl)\n                    added.append(f\"{rule['id']}: {inst} ⇒ {concl}\")\n        if not added:\n            break\n        log.append(f\"第 {step} 轮: \" + \"; \".join(added))\n    log.append(f\"目标 {data['goal']}: {'✓' if data['goal'] in known else '✗'}\")\n    return log\n\n\ndef backward_chain() -> list[str]:\n    data = load_rules()\n    goal = data[\"goal\"]\n    log = [f\"目标: {goal}\"]\n    sub = goal.replace(\"终有一死\", \"会死\")\n    log.append(f\"子目标: {sub} (R2)\")\n    sub2 = sub.replace(\"会死\", \"人\")\n    log.append(f\"子目标: {sub2} (R1)\")\n    log.append(f\"匹配事实: {sub2 in data['facts']} → 证明成立\")\n    return log\n\n\ndef backward_chain_table() -> pd.DataFrame:\n    data = load_rules()\n    goal = data[\"goal\"]\n    sub = goal.replace(\"终有一死\", \"会死\")\n    sub2 = sub.replace(\"会死\", \"人\")\n    return pd.DataFrame(\n        [\n            {\"步\": 1, \"当前目标\": goal, \"使用规则\": \"R2\", \"下一子目标\": sub, \"结果\": \"继续证明\"},\n            {\"步\": 2, \"当前目标\": sub, \"使用规则\": \"R1\", \"下一子目标\": sub2, \"结果\": \"继续证明\"},\n            {\"步\": 3, \"当前目标\": sub2, \"使用规则\": \"事实库\", \"下一子目标\": \"—\", \"结果\": \"证明成立\"},\n        ]\n    )\n\n\ndef build_adj(kg: dict) -> dict[str, list[tuple[str, str]]]:\n    adj: dict[str, list[tuple[str, str]]] = defaultdict(list)\n    for h, r, t in kg[\"edges\"]:\n        adj[h].append((r, t))\n    return adj\n\n\ndef graph_multihop() -> list[str]:\n    kg = load_kg()\n    adj = build_adj(kg)\n    works = [t for rel, t in adj[\"鲁迅\"] if rel == \"创作\"]\n    lines = [f\"鲁迅创作: {', '.join(works)}\"]\n    venues = {w: [t for rel, t in adj[w] if rel == \"发表于\"] for w in works}\n    for w, vs in venues.items():\n        lines.append(f\"{w} 发表于: {', '.join(vs)}\")\n    ans = kg[\"query\"][\"answer_y\"]\n    ok = all(ans in vs for vs in venues.values())\n    lines.append(f\"共同发表于 {ans}: {'✓' if ok else '✗'}\")\n    return lines\n\n\ndef kg_query_table() -> pd.DataFrame:\n    kg = load_kg()\n    return pd.DataFrame(\n        [\n            {\"项目\": \"查询模板\", \"内容\": kg[\"query\"][\"pattern\"]},\n            {\"项目\": \"答案 Y\", \"内容\": kg[\"query\"][\"answer_y\"]},\n        ]\n    )\n\n\ndef graph_edges_table() -> pd.DataFrame:\n    kg = load_kg()\n    return pd.DataFrame([{\"头实体\": h, \"关系\": r, \"尾实体\": t} for h, r, t in kg[\"edges\"]])\n\n\ndef entity_out_edges(entity: str) -> pd.DataFrame:\n    kg = load_kg()\n    adj = build_adj(kg)\n    return pd.DataFrame([{\"实体\": entity, \"关系\": r, \"指向\": t} for r, t in adj[entity]])\n\n\ndef graph_multihop_table() -> pd.DataFrame:\n    kg = load_kg()\n    adj = build_adj(kg)\n    works = [t for rel, t in adj[\"鲁迅\"] if rel == \"创作\"]\n    rows = [{\"跳数\": 1, \"输入\": \"鲁迅\", \"关系约束\": \"创作\", \"候选\": \"；\".join(works)}]\n    venues = {w: [t for rel, t in adj[w] if rel == \"发表于\"] for w in works}\n    for w, vs in venues.items():\n        rows.append({\"跳数\": 2, \"输入\": w, \"关系约束\": \"发表于\", \"候选\": \"；\".join(vs)})\n    ans = kg[\"query\"][\"answer_y\"]\n    ok = all(ans in vs for vs in venues.values())\n    rows.append({\"跳数\": \"汇总\", \"输入\": \"所有作品\", \"关系约束\": \"共同 Y\", \"候选\": f\"{ans} {'✓' if ok else '✗'}\"})\n    return pd.DataFrame(rows)\n\n\ndef codelens_forward_chain(max_steps: int = 10) -> list:\n    \"\"\"前向链每一轮：已知事实集 + 新推导。\"\"\"\n    from common.codelens import Frame\n\n    data = load_rules()\n    known = set(data[\"facts\"])\n    frames = [\n        Frame(0, \"known = facts\", \"初始化\", {\"known\": sorted(known), \"goal\": data[\"goal\"]}),\n    ]\n    step = 0\n    for round_i in range(1, max_steps + 1):\n        added = []\n        for rule in data[\"rules\"]:\n            cond = rule[\"if\"][0]\n            var = cond[cond.index(\"(\") + 1 : cond.index(\")\")]\n            inst = cond.replace(f\"({var})\", \"(苏格拉底)\")\n            if inst in known:\n                concl = rule[\"then\"].replace(f\"({var})\", \"(苏格拉底)\")\n                if concl not in known:\n                    known.add(concl)\n                    added.append(f\"{rule['id']}: {inst} ⇒ {concl}\")\n        if not added:\n            break\n        step += 1\n        frames.append(\n            Frame(\n                step,\n                f\"round {round_i} scan rules\",\n                f\"第 {round_i} 轮触发 {len(added)} 条规则\",\n                {\"新增\": added, \"known\": sorted(known)},\n            )\n        )\n    frames.append(\n        Frame(\n            step + 1,\n            \"check goal\",\n            \"检查目标是否在 known 中\",\n            {\"goal\": data[\"goal\"], \" proved\": data[\"goal\"] in known},\n        )\n    )\n    return frames\n\n\ndef forward_chain_table() -> pd.DataFrame:\n    rows = []\n    for frame in codelens_forward_chain():\n        added = frame.state.get(\"新增\", [])\n        known = frame.state.get(\"known\", [])\n        rows.append(\n            {\n                \"步\": frame.step,\n                \"动作\": frame.narrative,\n                \"新增事实\": \"；\".join(added) if isinstance(added, list) else \"\",\n                \"事实数\": len(known) if isinstance(known, list) else \"\",\n                \"目标成立\": frame.state.get(\" proved\", \"\"),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef animate_forward_chain() -> None:\n    from common.viz_anim import animate_items_row\n\n    frames = codelens_forward_chain()\n    snaps = []\n    for f in frames:\n        known = f.state.get(\"known\", [])\n        if isinstance(known, (list, tuple)):\n            items = [str(x) for x in known]\n        else:\n            items = []\n        new_items = f.state.get(\"新增\", [])\n        highlight = None\n        if new_items and isinstance(new_items, list) and new_items:\n            highlight = str(new_items[-1]).split(\":\")[-1].strip() if \":\" in str(new_items[-1]) else None\n        snaps.append(\n            {\n                \"step\": f.step,\n                \"items\": items,\n                \"highlight\": highlight,\n                \"action\": f.narrative,\n                \"extra\": \"; \".join(str(x) for x in new_items) if new_items else \"\",\n            }\n        )\n    animate_items_row(snaps, title=\"Known facts (forward chain)\")\n\n\ndef path_ranking() -> None:\n    kg = load_kg()\n    scores = kg[\"path_scores\"]\n    print(\"| 路径 | 得分 |\")\n    print(\"|------|------|\")\n    for path, sc in scores.items():\n        print(f\"| {path} | {sc} |\")\n    print(f\"\\n最高分路径: {max(scores, key=scores.get)}\")\n\n\ndef path_ranking_table() -> pd.DataFrame:\n    kg = load_kg()\n    rows = [{\"路径\": path, \"得分\": score} for path, score in kg[\"path_scores\"].items()]\n    return pd.DataFrame(rows).sort_values(\"得分\", ascending=False)\n\n\ndef plot_path_ranking() -> None:\n    table = path_ranking_table()\n    fig, ax = plt.subplots(figsize=(7, 2.6))\n    ax.barh(table[\"路径\"], table[\"得分\"], color=\"#0d6b62\")\n    ax.invert_yaxis()\n    ax.set_xlabel(\"score\")\n    ax.set_title(\"KG path ranking\")\n    plt.tight_layout()\n    plt.show()\n",
  "labs/common/campus_graph.json": "{\n  \"goal\": \"c1\",\n  \"start\": \"x\",\n  \"nodes\": {\n    \"x\": { \"name\": \"校门口\", \"h\": 7 },\n    \"c2\": { \"name\": \"超市\", \"h\": 1 },\n    \"j\": { \"name\": \"教学楼\", \"h\": 4 },\n    \"s2\": { \"name\": \"实验楼\", \"h\": 4 },\n    \"s1\": { \"name\": \"食堂\", \"h\": 3 },\n    \"t\": { \"name\": \"图书馆\", \"h\": 2 },\n    \"c1\": { \"name\": \"操场\", \"h\": 0 }\n  },\n  \"edges\": [\n    { \"from\": \"x\", \"to\": \"c2\", \"cost\": 7 },\n    { \"from\": \"x\", \"to\": \"j\", \"cost\": 2 },\n    { \"from\": \"x\", \"to\": \"s1\", \"cost\": 2 },\n    { \"from\": \"j\", \"to\": \"s2\", \"cost\": 4 },\n    { \"from\": \"s2\", \"to\": \"s1\", \"cost\": 1 },\n    { \"from\": \"s1\", \"to\": \"t\", \"cost\": 3 },\n    { \"from\": \"s1\", \"to\": \"c1\", \"cost\": 6 },\n    { \"from\": \"t\", \"to\": \"c1\", \"cost\": 2 }\n  ],\n  \"expected\": {\n    \"dfs\": { \"path\": [\"x\", \"j\", \"s2\", \"s1\", \"c1\"], \"steps\": 4, \"cost\": 13 },\n    \"bfs\": { \"path\": [\"x\", \"s1\", \"c1\"], \"steps\": 2, \"cost\": 8 },\n    \"ucs\": { \"path\": [\"x\", \"s1\", \"t\", \"c1\"], \"steps\": 3, \"cost\": 7 },\n    \"greedy\": { \"path\": [\"x\", \"s1\", \"c1\"], \"steps\": 2, \"cost\": 8 },\n    \"astar\": { \"path\": [\"x\", \"s1\", \"t\", \"c1\"], \"steps\": 3, \"cost\": 7 }\n  }\n}\n",
  "labs/common/ch6_kg.json": "{\n  \"nodes\": [\"鲁迅\", \"狂人日记\", \"呐喊\", \"文学周报社\", \"茅盾文学奖\", \"莫言\", \"蛙\", \"红高粱\", \"典藏\", \"电影\", \"金熊奖\"],\n  \"edges\": [\n    [\"鲁迅\", \"创作\", \"狂人日记\"],\n    [\"鲁迅\", \"创作\", \"呐喊\"],\n    [\"狂人日记\", \"发表于\", \"文学周报社\"],\n    [\"呐喊\", \"发表于\", \"文学周报社\"],\n    [\"狂人日记\", \"获得\", \"茅盾文学奖\"],\n    [\"莫言\", \"创作\", \"蛙\"],\n    [\"莫言\", \"创作\", \"红高粱\"],\n    [\"蛙\", \"获得\", \"茅盾文学奖\"],\n    [\"红高粱\", \"入选\", \"典藏\"],\n    [\"红高粱\", \"改编\", \"电影\"],\n    [\"电影\", \"获得\", \"金熊奖\"]\n  ],\n  \"query\": {\n    \"pattern\": [\"鲁迅\", \"创作\", \"?X\", \"?X\", \"发表于\", \"?Y\"],\n    \"answer_y\": \"文学周报社\"\n  },\n  \"path_scores\": {\n    \"蛙→茅盾文学奖\": 3,\n    \"红高粱→典藏\": 2,\n    \"红高粱→电影→金熊奖\": 3\n  }\n}\n",
  "labs/common/ch6_rules.json": "{\n  \"facts\": [\"人(苏格拉底)\"],\n  \"rules\": [\n    { \"id\": \"R1\", \"if\": [\"人(X)\"], \"then\": \"会死(X)\" },\n    { \"id\": \"R2\", \"if\": [\"会死(X)\"], \"then\": \"终有一死(X)\" }\n  ],\n  \"goal\": \"终有一死(苏格拉底)\"\n}\n",
  "labs/common/codelens.py": "\"\"\"CodeLens-style execution frames — print every variable change.\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Any\n\n\n@dataclass\nclass Frame:\n    step: int\n    line: str\n    narrative: str\n    state: dict[str, Any] = field(default_factory=dict)\n\n    def print(self) -> None:\n        print(f\"── Step {self.step} ── {self.narrative}\")\n        print(f\"   执行: {self.line}\")\n        for k, v in self.state.items():\n            print(f\"   {k} = {v!r}\")\n\n\ndef print_frames(frames: list[Frame], start: int = 0, stop: int | None = None) -> None:\n    for f in frames[start:stop]:\n        f.print()\n        print()\n\n\ndef frames_to_table(frames: list[Frame], keys: list[str]) -> \"pd.DataFrame\":\n    import pandas as pd\n\n    rows = []\n    for f in frames:\n        row = {\"步\": f.step, \"说明\": f.narrative}\n        for k in keys:\n            row[k] = f.state.get(k, \"\")\n        rows.append(row)\n    return pd.DataFrame(rows)\n",
  "labs/common/luxun_bpe.json": "{\n  \"corpus_hint\": \"鲁迅 写 了 狂人 日记\",\n  \"initial_tokens\": [\"鲁\", \"迅\", \"写\", \"了\", \"狂\", \"人\", \"日\", \"记\"],\n  \"merges\": [\n    { \"pair\": [\"日\", \"记\"], \"count\": 12, \"result\": \"日记\", \"tokens_after\": [\"鲁\", \"迅\", \"写\", \"了\", \"狂\", \"人\", \"日记\"] },\n    { \"pair\": [\"狂\", \"人\"], \"count\": 8, \"result\": \"狂人\", \"tokens_after\": [\"鲁\", \"迅\", \"写\", \"了\", \"狂人\", \"日记\"] },\n    { \"pair\": [\"鲁\", \"迅\"], \"count\": 6, \"result\": \"鲁迅\", \"tokens_after\": [\"鲁迅\", \"写\", \"了\", \"狂人\", \"日记\"] }\n  ],\n  \"final_tokens\": [\"鲁迅\", \"写\", \"了\", \"狂人\", \"日记\"]\n}\n",
  "labs/common/mpl_setup.py": "\"\"\"Matplotlib font setup for notebook figures.\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport warnings\nfrom pathlib import Path\n\nimport matplotlib as mpl\nimport matplotlib.font_manager as fm\n\n# Prefer one fixed CJK face in rendered site; keep local fallbacks for downloaded notebooks.\nCJK_FONT = \"Noto Sans CJK SC\"\n\n_FONT_PATHS = [\n    \"/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc\",\n    \"/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc\",\n    \"/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc\",\n    \"/System/Library/Fonts/PingFang.ttc\",\n    \"/System/Library/Fonts/STHeiti Light.ttc\",\n    \"/Library/Fonts/Arial Unicode.ttf\",\n]\n\n_CJK_NAMES = [\n    CJK_FONT,\n    \"Noto Sans SC\",\n    \"Source Han Sans SC\",\n    \"PingFang SC\",\n    \"Heiti SC\",\n    \"STHeiti\",\n    \"Arial Unicode MS\",\n    \"WenQuanYi Micro Hei\",\n]\n\n_CONFIGURED = False\n\n\ndef _register_font_files() -> str | None:\n    registered: list[str] = []\n    for path in _FONT_PATHS:\n        p = Path(path)\n        if not p.is_file():\n            continue\n        try:\n            fm.fontManager.addfont(str(p))\n            prop = fm.FontProperties(fname=str(p))\n            name = prop.get_name()\n            if name and name not in registered:\n                registered.append(name)\n        except Exception:\n            continue\n    return registered[0] if registered else None\n\n\ndef _find_cjk_font() -> str | None:\n    from_file = _register_font_files()\n    available = {f.name for f in fm.fontManager.ttflist}\n    if CJK_FONT in available:\n        return CJK_FONT\n    if from_file:\n        return from_file\n    for name in _CJK_NAMES:\n        if name in available:\n            return name\n    return None\n\n\ndef configure_matplotlib() -> None:\n    \"\"\"Notebook 首个绘图 cell 前调用一次。\"\"\"\n    global _CONFIGURED\n    if _CONFIGURED:\n        return\n    _CONFIGURED = True\n\n    logging.getLogger(\"matplotlib.font_manager\").setLevel(logging.ERROR)\n    warnings.filterwarnings(\"ignore\", message=\".*Glyph.*missing from font.*\")\n    warnings.filterwarnings(\"ignore\", message=\".*findfont.*\")\n\n    name = _find_cjk_font()\n    if name:\n        mpl.rcParams[\"font.sans-serif\"] = [name, \"DejaVu Sans\", \"sans-serif\"]\n        mpl.rcParams[\"font.family\"] = \"sans-serif\"\n    else:\n        mpl.rcParams[\"font.sans-serif\"] = [\"DejaVu Sans\", \"sans-serif\"]\n    mpl.rcParams[\"axes.unicode_minus\"] = False\n    mpl.rcParams[\"figure.dpi\"] = 100\n    mpl.rcParams[\"mathtext.default\"] = \"regular\"\n\n\ndef ascii_plot(text: str) -> str:\n    \"\"\"把常见数学符号换成 DejaVu 可显示的 ASCII（用于轴标签/GIF）。\"\"\"\n    repl = {\n        \"α\": \"alpha\",\n        \"β\": \"beta\",\n        \"γ\": \"gamma\",\n        \"δ\": \"delta\",\n        \"ε\": \"eps\",\n        \"σ\": \"sigma\",\n        \"ŷ\": \"y_hat\",\n        \"₁\": \"1\",\n        \"₂\": \"2\",\n        \"₃\": \"3\",\n        \"→\": \"->\",\n    }\n    out = text\n    for k, v in repl.items():\n        out = out.replace(k, v)\n    return out\n",
  "labs/common/notebook_helpers.py": "\"\"\"Shared helpers for pedagogical notebooks.\"\"\"\n\nfrom __future__ import annotations\n\nimport textwrap\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\n# Keep helper plots aligned with the rendered notebook font.\nplt.rcParams.update(\n    {\n        \"figure.figsize\": (7.5, 4.2),\n        \"font.size\": 11,\n        \"axes.unicode_minus\": False,\n        \"font.family\": \"sans-serif\",\n        \"font.sans-serif\": [\"Noto Sans CJK SC\", \"DejaVu Sans\", \"sans-serif\"],\n    }\n)\n\n\ndef repo_root() -> Path:\n    cwd = Path.cwd()\n    if (cwd / \"labs\").exists():\n        return cwd\n    if (cwd.parent / \"labs\").exists():\n        return cwd.parent\n    return cwd\n\n\nBOOTSTRAP = textwrap.dedent(\n    \"\"\"\n    import sys\n    from pathlib import Path\n    ROOT = Path.cwd()\n    if not (ROOT / \"labs\").exists() and (ROOT.parent / \"labs\").exists():\n        ROOT = ROOT.parent\n    if str(ROOT) not in sys.path:\n        sys.path.insert(0, str(ROOT))\n    \"\"\"\n).strip()\n\n\ndef bootstrap_code(extra: str = \"\") -> str:\n    return BOOTSTRAP + (\"\\n\" + extra.strip() if extra.strip() else \"\")\n\n\ndef show_df(df: pd.DataFrame, title: str = \"\") -> None:\n    if title:\n        print(title)\n    print(df.to_string(index=False))\n\n\ndef plot_line_series(\n    xs: list,\n    ys: list,\n    *,\n    title: str,\n    xlabel: str = \"\",\n    ylabel: str = \"\",\n    markers: bool = True,\n) -> None:\n    fig, ax = plt.subplots()\n    ax.plot(xs, ys, marker=\"o\" if markers else None, linewidth=2)\n    ax.set_title(title)\n    if xlabel:\n        ax.set_xlabel(xlabel)\n    if ylabel:\n        ax.set_ylabel(ylabel)\n    ax.grid(True, alpha=0.3)\n    plt.tight_layout()\n    plt.show()\n\n\ndef plot_scatter_labeled(\n    points: np.ndarray,\n    labels: np.ndarray | None,\n    *,\n    title: str,\n    label_names: dict[int, str] | None = None,\n) -> None:\n    fig, ax = plt.subplots()\n    if labels is None:\n        ax.scatter(points[:, 0], points[:, 1], s=60, c=\"#0d6b62\")\n    else:\n        for lab in np.unique(labels):\n            mask = labels == lab\n            name = (label_names or {}).get(int(lab), f\"类 {lab}\")\n            ax.scatter(points[mask, 0], points[mask, 1], s=60, label=name)\n        ax.legend()\n    ax.set_title(title)\n    ax.grid(True, alpha=0.3)\n    plt.tight_layout()\n    plt.show()\n"
}

ROOT = Path.cwd() / "_ai_thinking_labs_inline_runtime"
for rel, source in INLINE_RUNTIME_FILES.items():
    target = ROOT / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")

missing = []
for module, package in [
    ("numpy", "numpy>=1.24"),
    ("pandas", "pandas>=2.0"),
    ("matplotlib", "matplotlib>=3.7"),
    ("scipy", "scipy>=1.10"),
    ("sklearn", "scikit-learn>=1.3"),
]:
    if importlib.util.find_spec(module) is None:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "labs"))
sys.path.insert(0, str(ROOT / "labs" / "ch06"))
import matplotlib.pyplot as plt
from common.mpl_setup import configure_matplotlib
configure_matplotlib()
from IPython.display import display, Image
print("runtime ready:", ROOT)
from reasoning import *

runtime ready: /Users/zhesun/Desktop/Fudan/phd/vibe coding/notebooks/_ai_thinking_labs_inline_runtime


In [2]:
# 保存本步骤变量，供后续代码单元使用。
data = load_rules()

In [3]:
# 显示当前实验结果表。
display(facts_goal_table())

,项目,内容
0,初始事实,人(苏格拉底)
1,证明目标,终有一死(苏格拉底)


In [4]:
# 显示当前实验结果表。
display(rules_table())

,规则,IF,THEN
0,R1,人(X),会死(X)
1,R2,会死(X),终有一死(X)


## 1. 前向链

In [5]:
# 显示当前实验结果表。
display(forward_chain_table())

,步,动作,新增事实,事实数,目标成立
0,0,初始化,,1,
1,1,第 1 轮触发 2 条规则,R1: 人(苏格拉底) ⇒ 会死(苏格拉底)；R2: 会死(苏格拉底) ⇒ 终有一死(苏格拉底),3,
2,2,检查目标是否在 known 中,,0,True


## 2. 后向链

In [6]:
# 显示当前实验结果表。
display(backward_chain_table())

,步,当前目标,使用规则,下一子目标,结果
0,1,终有一死(苏格拉底),R2,会死(苏格拉底),继续证明
1,2,会死(苏格拉底),R1,人(苏格拉底),继续证明
2,3,人(苏格拉底),事实库,—,证明成立
